# Test document extraction

Use this notebook to test `rag/extractors.py` before building chunking.

1. Copy a small `.txt`, `.pdf`, or `.docx` file into `data/uploads/`.
2. In VS Code, choose the **Python (.venv)** kernel.
3. Run the cells from top to bottom.

The notebook shows how many source sections were extracted, previews their cleaned text, and checks the metadata used later for chunking and citations.

In [1]:
from pathlib import Path
import sys

# VS Code usually starts notebooks in the project folder.
# This fallback also works when the notebook starts inside notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / 'rag').is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.extractors import extract_document

upload_folder = project_root / 'data' / 'uploads'
print(f'Project folder: {project_root}')
print(f'Upload folder: {upload_folder}')

Project folder: /Users/ememusoh/Desktop/Project
Upload folder: /Users/ememusoh/Desktop/Project/data/uploads


In [2]:
# Find files that the current extractor supports.
supported_extensions = {'.txt', '.pdf', '.docx'}
available_files = sorted(
    file_path
    for file_path in upload_folder.iterdir()
    if file_path.is_file() and file_path.suffix.lower() in supported_extensions
)

if available_files:
    for index, file_path in enumerate(available_files):
        print(f'{index}: {file_path.name}')
else:
    print('No supported files found. Add a TXT, PDF, or DOCX file to data/uploads/.')

0: Project_Proposal.docx


In [3]:
# Change 0 to test another file from the numbered list above.
if not available_files:
    raise FileNotFoundError('Add a supported document to data/uploads/, then run the notebook again.')

file_to_test = available_files[0]
sections = extract_document(file_to_test)

print(f'Testing: {file_to_test.name}')
print(f'Extracted sections: {len(sections)}')

Testing: Project_Proposal.docx
Extracted sections: 39


In [4]:
# Preview the first three sections. These are the inputs for the future chunking step.
for section in sections[:3]:
    print('=' * 80)
    print(section['source_label'])
    print(section['text'][:500])

if not sections:
    print('No extractable text was found. This can happen with scanned PDFs or empty files.')

Project_Proposal.docx, paragraph 1
Project Proposal
Project_Proposal.docx, paragraph 2
Ask My Documents: A RAG System for Question Answering over Uploaded Documents
Project_Proposal.docx, paragraph 3
Student: Emem Usoh | Supervisor: Professor Dongyu Qiu | Duration: Summer 2 (6 weeks)


In [5]:
# Check that the metadata needed for citations and chunking is present.
required_keys = {
    'document_name', 'file_type', 'source_id', 'source_label',
    'source_type', 'source_number', 'text',
}

if sections:
    missing_keys = required_keys - sections[0].keys()
    if missing_keys:
        print(f'Missing metadata: {sorted(missing_keys)}')
    else:
        print('Metadata check passed.')
        print(sections[0])

Metadata check passed.
{'document_name': 'Project_Proposal.docx', 'document_path': '/Users/ememusoh/Desktop/Project/data/uploads/Project_Proposal.docx', 'file_type': 'docx', 'source_id': 'Project_Proposal.docx:paragraph:1', 'source_type': 'paragraph', 'source_number': 1, 'source_label': 'Project_Proposal.docx, paragraph 1', 'text': 'Project Proposal', 'paragraph_number': 1}


## What to check

- The number of sections makes sense for the document.
- PDF sections show page numbers; DOCX and TXT sections show paragraph information.
- The preview text is readable and does not contain distracting extra whitespace.
- Each section has a clear `source_label`, which will become the basis for citations.